# Evaluate Performance Using CatBoost

In [16]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from shap_select import shap_select
import xgboost as xgb
from xgboost import XGBRegressor
import optuna
from catboost import CatBoostRegressor

In [2]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [3]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


In [4]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance

In [5]:
train_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
2014,165.161,1310.0,11.0,204.30000,1012.728952,2907.0,5351.0,14948.5,12732.0,9699.0,...,0.000002,0.132912,5.350128,1825.0,1533.0,79989423.5,4.725000e+03,3.675000e+03,-316.669417,0.925926
1334,121.954,377.0,93.0,173.10000,1363.117917,2946.0,4692.0,15514.0,NaN,NaN,...,0.000251,0.186936,34.724747,1220.0,1159.0,72791688.0,1.819300e+04,1.502900e+04,-11.702699,0.869565
5898,81.145,179.6,20.0,133.30000,115.056783,2186.0,4889.0,14703.5,11544.5,12228.0,...,0.000093,0.254898,3.850726,1792.0,1728.0,71885411.5,1.683745e+06,1.515370e+06,-19.904327,0.933333
2314,245.918,758.0,183.0,168.40001,1296.518526,1623.0,3284.5,14641.0,13889.5,9824.0,...,0.000001,0.145714,0.000000,1782.0,1188.0,48088364.5,4.117773e+04,2.470664e+04,-35.600384,0.900000
5612,113.106,553.0,49.0,174.00000,952.729122,3244.0,5786.0,15102.0,13187.5,9932.0,...,0.000538,0.301439,10.106505,1495.0,1430.0,87380172.0,1.263721e+06,1.029699e+06,-5.551639,0.851852


In [6]:
train_df.shape

(6523, 33)

In [7]:
# Split train_df and test_df into X_train, X_test, y_train and y_test

X_train = train_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_df[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

### Preprocess Data - Catboost doesn't require Imputation or Scaling, can essentially skip this step.

## **Feature Selection** 

In [8]:
# Will need to do different feature selection again for CatBoost since preprocessing steps are different than SVR.
# Conduct feature selection using shap_select again.

def perform_feature_selection(X_train, y_train):
    results_dict = {}

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_train, test_size=0.2)
    X_te_df = pd.DataFrame(X_te, columns=X_train.columns, index=y_te.index)
    
    
    for target in y_train.columns:
        model = XGBRegressor(n_estimators=1000, verbosity = 0, eval_metric='rmse', objective='reg:squarederror')
        model.fit(X_tr, y_tr[target], eval_set=[(X_te, y_te[target])])

        selected_df = shap_select(model, X_te_df, y_te[target], task="regression", threshold=0.05)
        results_dict[target] = selected_df

    return results_dict  

In [15]:
results = perform_feature_selection(X_train, y_train)

[0]	validation_0-rmse:58.85491
[1]	validation_0-rmse:49.90454
[2]	validation_0-rmse:43.98121
[3]	validation_0-rmse:39.98657
[4]	validation_0-rmse:37.09691
[5]	validation_0-rmse:35.05922
[6]	validation_0-rmse:34.03959
[7]	validation_0-rmse:33.39492
[8]	validation_0-rmse:32.66944
[9]	validation_0-rmse:31.96229
[10]	validation_0-rmse:31.51794
[11]	validation_0-rmse:31.27404
[12]	validation_0-rmse:31.06127
[13]	validation_0-rmse:30.93100
[14]	validation_0-rmse:30.66770
[15]	validation_0-rmse:30.61829
[16]	validation_0-rmse:30.24094
[17]	validation_0-rmse:30.12736
[18]	validation_0-rmse:29.86075
[19]	validation_0-rmse:29.73168
[20]	validation_0-rmse:29.58639
[21]	validation_0-rmse:29.61858
[22]	validation_0-rmse:29.61217
[23]	validation_0-rmse:29.52397
[24]	validation_0-rmse:29.41544
[25]	validation_0-rmse:29.36100
[26]	validation_0-rmse:29.31721
[27]	validation_0-rmse:29.21089
[28]	validation_0-rmse:29.19532
[29]	validation_0-rmse:29.12903
[30]	validation_0-rmse:28.97629
[31]	validation_0-

In [17]:
results

{'Total Alkalinity':                         feature name    t-value  stat.significance  \
 0                 cec_pH_interaction  15.673946       7.361350e-51   
 1                   skin_temperature  15.164197       5.871305e-48   
 2                  flow_accumulation  12.205200       1.616905e-32   
 3                          elevation   7.912707       5.352284e-15   
 4                                nir   7.869746       7.430035e-15   
 5                             swir16   6.080586       1.571873e-09   
 6            total_precipitation_sum   5.280180       1.511775e-07   
 7               NDVI_LST_interaction   4.425181       1.044363e-05   
 8                              MNDWI   3.663210       2.591170e-04   
 9              total_evaporation_sum   3.185135       1.481363e-03   
 10          Land Surface Temperature   2.843609       4.530778e-03   
 11                    temperature_2m   2.369782       1.794457e-02   
 12             volumetric_soil_water   1.667729       9.

In [9]:
total_alk_feats = ["cec_pH_interaction", "skin_temperature", "flow_accumulation", "elevation", "nir",
                   "swir16", "total_precipitation_sum", "NDVI_LST_interaction", "MNDWI", "total_evaporation_sum",
                   "Land Surface Temperature", "temperature_2m"]

len(total_alk_feats)

12

In [10]:
elec_cond_feats = [
    "skin_temperature",
    "NDVI",
    "elevation",
    "phosphorous",
    "total_evaporation_sum",
    "flow_acc_clay_interaction",
    "clay",
    "pH",
    "precipitation",
    "phosphorous_pH_interaction",
    "pet",
    "cec_clay_ratio",
    "cec_pH_interaction"
]

len(elec_cond_feats)

13

In [11]:
drp_feats = [
    "phosphorous_pH_interaction",
    "cec_pH_interaction",
    "pet",
    "flow_accumulation",
    "total_precipitation_sum",
    "elevation",
    "EVI",
    "nir",
    "flow_acc_phosphorous_interaction",
    "Land Surface Temperature",
    "cec_clay_ratio"
]

len(drp_feats)

11

In [14]:
selected_feats = list(set(total_alk_feats + elec_cond_feats + drp_feats))

len(selected_feats)

23

## **Hyperparamter Optimization**

In [15]:
# Define new X_train and X_test based on feature selection results

X_train_fe = X_train[selected_feats]
X_test_fe = X_test[selected_feats]

X_train_fe.head()

,phosphorous_pH_interaction,cec_clay_ratio,cec_pH_interaction,NDVI_LST_interaction,nir,flow_accumulation,phosphorous,total_precipitation_sum,pet,pH,...,skin_temperature,MNDWI,elevation,total_evaporation_sum,EVI,flow_acc_clay_interaction,temperature_2m,Land Surface Temperature,flow_acc_phosphorous_interaction,swir16
2014,1533.0,0.925926,1825.0,79989423.5,12732.0,175.000000,21.0,0.000002,204.30000,73.0,...,292.891655,-0.092470,1012.728952,-0.000998,2907.0,4.725000e+03,291.355673,14948.5,3.675000e+03,11675.5
1334,1159.0,0.869565,1220.0,72791688.0,NaN,791.000000,19.0,0.000251,173.10000,61.0,...,294.242273,NaN,1363.117917,-0.002943,2946.0,1.819300e+04,292.974353,15514.0,1.502900e+04,NaN
5898,1728.0,0.933333,1792.0,71885411.5,11544.5,56124.822222,27.0,0.000093,133.30000,64.0,...,288.487464,0.143712,115.056783,-0.001870,2186.0,1.683745e+06,288.729935,14703.5,1.515370e+06,9155.0
2314,1188.0,0.900000,1782.0,48088364.5,13889.5,1372.590909,18.0,0.000001,168.40001,66.0,...,283.771401,-0.263596,1296.518526,-0.000075,1623.0,4.117773e+04,282.983824,14641.0,2.470664e+04,16857.0
5612,1430.0,0.851852,1495.0,87380172.0,13187.5,46804.487179,22.0,0.000538,174.00000,65.0,...,294.255641,-0.051068,952.729122,-0.002991,3244.0,1.263721e+06,293.876384,15102.0,1.029699e+06,11001.0


In [ ]:
# Next steps: perform hyperparameter optimization using Optuna. Then, using optimal
# hyperparameters retrain model for final testing.

def objective(trial):
    iterations = trial.suggest_int("iterations", 300, 1500)
    depth = trial.suggest_int("depth", 4, 10)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    l2_leaf_reg =  trial.suggest_float("l2_leaf_reg", 1, 10)
    bagging_temperature =  trial.suggest_float("bagging_temperature", 0, 5)
    random_strength =  trial.suggest_float("random_strength", 0, 5)


    catboost = MultiOutputRegressor(CatBoostRegressor(iterations=iterations, learning_rate=learning_rate,
                                    depth=depth, l2_leaf_reg=l2_leaf_reg, 
                                    bagging_temperature=bagging_temperature,
                                    random_strength=random_strength, allow_writing_files=False, early_stopping_rounds=50))
    
    score = cross_val_score(catboost, X_train_fe, y_train, cv=5, scoring="r2", n_jobs=-1).mean()

    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

In [27]:
study.best_trial

FrozenTrial(number=26, state=<TrialState.COMPLETE: 1>, values=[0.7839309668625903], datetime_start=datetime.datetime(2026, 3, 12, 16, 26, 1, 42046), datetime_complete=datetime.datetime(2026, 3, 12, 16, 26, 27, 576874), params={'iterations': 1322, 'depth': 7, 'learning_rate': 0.03653435268116288, 'l2_leaf_reg': 1.0357053375540382, 'bagging_temperature': 3.9705584233937956, 'random_strength': 0.24148881154228752}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'iterations': IntDistribution(high=1500, log=False, low=300, step=1), 'depth': IntDistribution(high=10, log=False, low=4, step=1), 'learning_rate': FloatDistribution(high=0.3, log=False, low=0.01, step=None), 'l2_leaf_reg': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'bagging_temperature': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'random_strength': FloatDistribution(high=5.0, log=False, low=0.0, step=None)}, trial_id=26, value=None)

In [33]:
study.best_params

{'iterations': 1322,
 'depth': 7,
 'learning_rate': 0.03653435268116288,
 'l2_leaf_reg': 1.0357053375540382,
 'bagging_temperature': 3.9705584233937956,
 'random_strength': 0.24148881154228752}

In [29]:
study.best_value

0.7839309668625903

Results for CatBoost are Above, Regression chaining doesnt work with Nan values so using multi_output_regression throughout here.

In [ ]:
## FINAL MODEL EVALUATION ON TEST SET TO SEE PERFORMANCE

cb = MultiOutputRegressor(CatBoostRegressor(iterations=1322, depth=7, learning_rate=0.037,
                        l2_leaf_reg=1.036, bagging_temperature=3.97, random_strength=0.241))

cb.fit(X_train_fe, y_train)
score = cb.score(X_test_fe, y_test)

score

0:	learn: 73.1112162	total: 55.4ms	remaining: 1m 13s
1:	learn: 71.5990099	total: 57.3ms	remaining: 37.8s
2:	learn: 70.1739701	total: 59ms	remaining: 25.9s
3:	learn: 68.7864992	total: 60.5ms	remaining: 19.9s
4:	learn: 67.4332789	total: 62.2ms	remaining: 16.4s
5:	learn: 66.1556674	total: 63.6ms	remaining: 14s
6:	learn: 64.9742106	total: 65.4ms	remaining: 12.3s
7:	learn: 63.8153822	total: 66.9ms	remaining: 11s
8:	learn: 62.7333356	total: 68.3ms	remaining: 9.97s
9:	learn: 61.6952620	total: 69.9ms	remaining: 9.17s
10:	learn: 60.6694272	total: 72.3ms	remaining: 8.61s
11:	learn: 59.6809725	total: 73.8ms	remaining: 8.05s
12:	learn: 58.7246726	total: 75.1ms	remaining: 7.57s
13:	learn: 57.8411109	total: 76.4ms	remaining: 7.14s
14:	learn: 57.0011256	total: 77.8ms	remaining: 6.78s
15:	learn: 56.1925357	total: 79.2ms	remaining: 6.46s
16:	learn: 55.3996431	total: 80.5ms	remaining: 6.18s
17:	learn: 54.6916757	total: 82.1ms	remaining: 5.94s
18:	learn: 53.9676527	total: 83.5ms	remaining: 5.72s
19:	lear

0.7820469130417637

In [37]:
## Predict Output for Submission Set

submission_df = pd.read_csv('../data/validation_set.csv')
submission_df.head()

,Unnamed: 0,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN,161.90001,780.299056,1628.0,...,0.000007,0.136892,0.000000,1742.0,1742.0,4.542096e+07,351115.714286,338111.428571,-95.760822,0.962963
1,1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,177.60000,279.351413,2933.0,...,0.000550,0.259777,27.521283,1495.0,1690.0,1.069767e+08,202225.000000,210314.000000,-3.653135,0.920000
2,2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN,158.40001,163.622679,4032.0,...,0.000174,0.232120,14.464016,1403.0,1525.0,1.034820e+08,121744.000000,126816.666667,-10.921542,0.958333
3,3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,130.00000,44.046365,4701.0,...,NaN,NaN,42.085633,1121.0,1416.0,1.123978e+08,18889.531915,18889.531915,NaN,0.791667
4,4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN,152.50000,333.249422,1498.0,...,0.000618,0.336999,30.467072,1674.0,1674.0,6.468512e+07,410099.609756,381816.878049,-4.473473,0.931034


In [38]:
check_df = submission_df.drop(columns=['Unnamed: 0', 'Latitude', 
'Longitude', 'Sample Date', 
"Total Alkalinity", "Electrical Conductance",
'Dissolved Reactive Phosphorus'])

check_df.head()

,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,swir16,swir22,NDMI,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,161.90001,780.299056,1628.0,3016.0,15060.0,15229.0,12868.0,14797.0,12421.0,0.014388,...,0.000007,0.136892,0.000000,1742.0,1742.0,4.542096e+07,351115.714286,338111.428571,-95.760822,0.962963
1,177.60000,279.351413,2933.0,7263.0,14729.0,NaN,NaN,NaN,NaN,NaN,...,0.000550,0.259777,27.521283,1495.0,1690.0,1.069767e+08,202225.000000,210314.000000,-3.653135,0.920000
2,158.40001,163.622679,4032.0,7036.0,14707.5,16221.0,9304.5,12536.5,9958.0,0.128123,...,0.000174,0.232120,14.464016,1403.0,1525.0,1.034820e+08,121744.000000,126816.666667,-10.921542,0.958333
3,130.00000,44.046365,4701.0,7474.5,15037.5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,42.085633,1121.0,1416.0,1.123978e+08,18889.531915,18889.531915,NaN,0.791667
4,152.50000,333.249422,1498.0,4257.0,15195.0,9125.0,11100.5,9455.0,8711.0,-0.017761,...,0.000618,0.336999,30.467072,1674.0,1674.0,6.468512e+07,410099.609756,381816.878049,-4.473473,0.931034


In [40]:
test_df = check_df[selected_feats]

test_df.head()

,phosphorous_pH_interaction,cec_clay_ratio,cec_pH_interaction,NDVI_LST_interaction,nir,flow_accumulation,phosphorous,total_precipitation_sum,pet,pH,...,skin_temperature,MNDWI,elevation,total_evaporation_sum,EVI,flow_acc_clay_interaction,temperature_2m,Land Surface Temperature,flow_acc_phosphorous_interaction,swir16
0,1742.0,0.962963,1742.0,4.542096e+07,15229.0,13004.285714,26.0,0.000007,161.90001,67.0,...,288.386807,-0.069727,780.299056,-0.000788,1628.0,351115.714286,287.626439,15060.0,338111.428571,14797.0
1,1690.0,0.920000,1495.0,1.069767e+08,NaN,8089.000000,26.0,0.000550,177.60000,65.0,...,288.667907,NaN,279.351413,-0.002014,2933.0,202225.000000,287.174347,14729.0,210314.000000,NaN
2,1525.0,0.958333,1403.0,1.034820e+08,16221.0,5072.666667,25.0,0.000174,158.40001,61.0,...,291.107708,-0.147979,163.622679,-0.001915,4032.0,121744.000000,291.330903,14707.5,126816.666667,12536.5
3,1416.0,0.791667,1121.0,1.123978e+08,NaN,787.063830,24.0,NaN,130.00000,59.0,...,NaN,NaN,44.046365,NaN,4701.0,18889.531915,NaN,15037.5,18889.531915,NaN
4,1674.0,0.931034,1674.0,6.468512e+07,9125.0,14141.365854,27.0,0.000618,152.50000,62.0,...,288.706654,0.080052,333.249422,-0.002771,1498.0,410099.609756,288.034482,15195.0,381816.878049,9455.0


In [41]:
vals = cb.predict(test_df)
val_df = pd.DataFrame(vals)
val_df.columns = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

val_df

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,124.621303,337.688026,21.330442
1,86.445012,425.564354,45.632628
2,61.420649,202.549960,21.500480
3,71.374396,322.545137,9.694899
4,88.838084,258.189106,18.876887
...,...,...,...
195,59.829777,433.446571,27.188335
196,85.026952,367.828543,19.510188
197,93.389578,353.277355,31.694825
198,59.293472,367.635304,32.677361


In [42]:
submission = pd.DataFrame({
    'Longitude': submission_df['Longitude'],
    'Latitude': submission_df['Latitude'],
    'Sample Date': submission_df['Sample Date'],
    'Total Alkalinity': val_df['Total Alkalinity'],
    'Electrical Conductance': val_df['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': val_df['Dissolved Reactive Phosphorus']
})

submission

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,124.621303,337.688026,21.330442
1,26.077500,-33.329167,16-09-2015,86.445012,425.564354,45.632628
2,27.640028,-32.991639,07-05-2015,61.420649,202.549960,21.500480
3,24.439167,-34.096389,07-02-2012,71.374396,322.545137,9.694899
4,28.581667,-32.000556,01-10-2014,88.838084,258.189106,18.876887
...,...,...,...,...,...,...
195,25.386667,-33.771111,06-12-2012,59.829777,433.446571,27.188335
196,27.390750,-33.185361,04-09-2014,85.026952,367.828543,19.510188
197,27.822778,-32.043333,28-09-2015,93.389578,353.277355,31.694825
198,25.161389,-33.001667,08-01-2015,59.293472,367.635304,32.677361


In [44]:
submission.to_csv('../submission.csv', index=False)